In [1]:
#!pip3 list

Package                 Version
----------------------- ----------------
annotated-types         0.7.0
anyio                   4.11.0
appdirs                 1.4.4
argon2-cffi             21.1.0
attrs                   21.2.0
Automat                 20.2.0
Babel                   2.8.0
backcall                0.2.0
bcrypt                  3.2.0
beautifulsoup4          4.10.0
beniget                 0.4.1
bleach                  4.1.0
blinker                 1.4
Brotli                  1.0.9
certifi                 2020.6.20
chardet                 4.0.0
click                   8.0.3
cloud-init              25.2
colorama                0.4.4
command-not-found       0.3
configobj               5.0.6
constantly              15.1.0
cryptography            3.4.8
cycler                  0.11.0
dbus-python             1.2.18
decorator               4.4.2
defusedxml              0.7.1
distro                  1.7.0
distro-info             1.1+ubuntu0.2
entrypoints             0.4
filelock      

In [2]:
#!pip uninstall langchain-huggingface -y

Found existing installation: langchain-huggingface 1.0.1
Uninstalling langchain-huggingface-1.0.1:
  Successfully uninstalled langchain-huggingface-1.0.1


In [3]:
#!pip3 install langchain-huggingface

  Using cached langchain_huggingface-1.0.1-py3-none-any.whl.metadata (2.1 kB)
Using cached langchain_huggingface-1.0.1-py3-none-any.whl (27 kB)
    sys-platform (=="darwin") ; extra == 'objc'
                 ~^


In [2]:
#from langchain_community.llms import LlamaCpp
from pydantic import BaseModel, Field
from typing import List, Literal
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate


# Определяем структуру данных
class SentimentAnalysis(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"] = Field(
        description="Тональность отзыва: положительная, отрицательная или нейтральная"
    )
    confidence: float = Field(
        description="Уверенность в анализе от 0.0 до 1.0",
        ge=0.0, le=1.0
    )
    key_topics: List[str] = Field(
        description="Ключевые темы, упомянутые в отзыве",
        max_items=5
    )
    summary: str = Field(
        description="Краткое резюме отзыва в одном предложении",
        max_length=200
    )

# Создаем парсер
parser = JsonOutputParser(pydantic_object=SentimentAnalysis)

# Создаем умный шаблон
# prompt_template = PromptTemplate(
#     template="""Проанализируй отзыв: {review}

# {format_instructions}

# ТОЛЬКО JSON!""",
#     input_variables=["review"],
#     partial_variables={
#         "format_instructions": parser.get_format_instructions()  # Автомагия!
#     }
# )




# Инициализируем нейросеть
# Путь к файлу модели в формате GGUF
model_path = "Qwen/Qwen3-4B-Instruct-2507" #"Qwen3-8B-Q5_0.gguf"


# # Инициализация модели LlamaCpp с параметрами
# llm = LlamaCpp(
#     model_path=model_path,      # путь к файлу модели
#     temperature=0.0,           # степень случайности (0-1, где 0 - детерминировано)
#     max_tokens=1000,           # максимальное количество токенов в ответе
#     top_p=0.9,                 # параметр ядерной выборки
#     n_ctx=6000,                 # размер контекстного окна (сколько токенов "помнит" модель)
#     n_gpu_layers = 65

# )


/tmp/ipykernel_132863/1428672944.py:17: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  key_topics: List[str] = Field(


In [3]:
from langchain_community.llms import VLLM

llm = VLLM(model=model_path,
  #         trust_remote_code=True,  # mandatory for hf models
            temperature = 0.0, 
           vllm_kwargs={"max_model_len": 4096}
                          
)

INFO 11-18 19:00:54 [__init__.py:216] Automatically detected platform cuda.
INFO 11-18 19:00:56 [utils.py:233] non-default args: {'max_model_len': 4096, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B-Instruct-2507'}
INFO 11-18 19:00:57 [model.py:547] Resolved architecture: Qwen3ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 11-18 19:00:57 [model.py:1510] Using max model len 4096
INFO 11-18 19:00:59 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=132961) INFO 11-18 19:01:00 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=132961) INFO 11-18 19:01:00 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='Qwen/Qwen3-4B-Instruct-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Instruct-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_prope

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore_DP0 pid=132961) INFO 11-18 19:01:06 [default_loader.py:267] Loading weights took 1.85 seconds
(EngineCore_DP0 pid=132961) INFO 11-18 19:01:06 [gpu_model_runner.py:2653] Model loading took 7.6065 GiB and 2.920195 seconds
(EngineCore_DP0 pid=132961) INFO 11-18 19:01:12 [backends.py:548] Using cache directory: /root/.cache/vllm/torch_compile_cache/b78413f424/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=132961) INFO 11-18 19:01:12 [backends.py:559] Dynamo bytecode transform time: 5.92 s
(EngineCore_DP0 pid=132961) INFO 11-18 19:01:16 [backends.py:164] Directly load the compiled graph(s) for dynamic shape from the cache, took 2.719 s
(EngineCore_DP0 pid=132961) INFO 11-18 19:01:17 [monitor.py:34] torch.compile takes 5.92 s in total
(EngineCore_DP0 pid=132961) INFO 11-18 19:01:18 [gpu_worker.py:298] Available KV cache memory: 12.09 GiB
(EngineCore_DP0 pid=132961) INFO 11-18 19:01:18 [kv_cache_utils.py:1087] GPU KV cache size: 88,064 tokens
(EngineCore_DP0 pid=13

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:02<00:00, 22.90it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 22.75it/s]


(EngineCore_DP0 pid=132961) INFO 11-18 19:01:24 [gpu_model_runner.py:3480] Graph capturing finished in 5 secs, took 0.74 GiB
(EngineCore_DP0 pid=132961) INFO 11-18 19:01:24 [core.py:210] init engine (profile, create kv cache, warmup model) took 17.52 seconds
INFO 11-18 19:01:25 [llm.py:306] Supported_tasks: ['generate']


In [4]:
prompt_template = PromptTemplate(
#     template="""
# Ты — генератор JSON. Верни только json.
# JSON должен строго соответствовать схеме:

# {format_instructions}

# Вот отзыв для анализа:
# "{review}"
# <|eot_id|>
# """,
        template="""Проанализируй отзыв: {review}

{format_instructions}

ТОЛЬКО JSON!  <|eot_id|>""",
    input_variables=["review"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)


In [5]:
prompt_template

PromptTemplate(input_variables=['review'], input_types={}, partial_variables={'format_instructions': 'STRICT OUTPUT FORMAT:\n- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.\n- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).\n- Do not prepend or append any text (e.g., do not write "Here is the JSON:").\n- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.\n\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output

In [6]:
# Тестовый отзыв
#review = "Товар отличный, быстрая доставка! Очень доволен покупкой."
review = "Судя по отзывам товар хороший, но мне не понравилось, "

print("=== ПОШАГОВОЕ ВЫПОЛНЕНИЕ ===")

# Шаг 1: Применяем шаблон
print("Применяем PromptTemplate")
prompt_value = prompt_template.invoke({"review": review})
print(f"Тип: {type(prompt_value)}")

# Посмотрим на готовый промпт
prompt_text = prompt_value.to_string()
print("Готовый промпт:")
print(prompt_text[:200] + "...")  # Первые 200 символов
print()

# Шаг 2: Отправляем в нейросеть
print("Отправляем в нейросеть")
llm_response = llm.invoke(prompt_value)


=== ПОШАГОВОЕ ВЫПОЛНЕНИЕ ===
Применяем PromptTemplate
Тип: <class 'langchain_core.prompt_values.StringPromptValue'>
Готовый промпт:
Проанализируй отзыв: Судя по отзывам товар хороший, но мне не понравилось, 

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explana...

Отправляем в нейросеть


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [7]:
llm_response

' \n\n{"sentiment": "negative", "confidence": 0.95, "key_topics": ["качество товара", "впечатления от покупки"], "summary": "Несмотря на то, что товар в целом оценивается положительно, пользователь не нашел в нем того, что ожидал, и выражает отрицательное отношение к покупке."}'

In [8]:
# try:
#     json_text = llm_response.split("<json>")[1].split("</json>")[0].strip()
# except IndexError:
#     raise ValueError("Модель не вернула JSON в тегах <json>…</json>")

parsed_result = parser.invoke(llm_response)
print(parsed_result)


{'sentiment': 'negative', 'confidence': 0.95, 'key_topics': ['качество товара', 'впечатления от покупки'], 'summary': 'Несмотря на то, что товар в целом оценивается положительно, пользователь не нашел в нем того, что ожидал, и выражает отрицательное отношение к покупке.'}


In [ ]:
print(f"Тип ответа: {type(llm_response)}")
print(f"Ответ: {llm_response}")
print()

# Шаг 3: Парсим JSON
print("Парсим JSON")
parsed_result = parser.invoke(llm_response)
print(f"Тип результата: {type(parsed_result)}")
print("Структурированные данные:")
for key, value in parsed_result.items():
    print(f"  {key}: {value}")

In [ ]:
# --- Ввод промпта вручную ---
review_text = input("Введите отзыв: ")

# Создаём готовый промпт из шаблона
prompt = prompt_template.format(review=review_text)

# Генерируем ответ модели
raw_output = llm.invoke(prompt)

print("\nСЫРОЙ ОТВЕТ МОДЕЛИ:")
print(raw_output)

# Парсим JSON → превращаем в Python-объект
result = parser.parse(raw_output)

print("\nСТРУКТУРИРОВАННЫЕ ДАННЫЕ:")
print(result)
